# Lopsided — Week 1 Starter Notebook
Load ISLES'24 perfusion CT, visualize the four blood-flow maps,
and extract left-vs-right asymmetry features.

Run cells top to bottom in Google Colab (same account as your Drive).

**Data note:** this public mirror stores every patient inside a `.parquet`
table (one patient per file), with each scan packed as compressed bytes.
So we don't look for loose `.nii` files — we open the table and read each
scan out of it, ONE patient at a time. That keeps memory flat and avoids
the "ran out of RAM" crash.

In [ ]:
# --- Install dependencies ---
!pip install -q nibabel huggingface_hub pyarrow pandas numpy matplotlib scipy

In [ ]:
# --- Mount Google Drive and create the project folder ---
from google.colab import drive
import os
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/isles_data'
os.makedirs(PROJECT, exist_ok=True)
print('Project folder:', PROJECT)

## Download the data (runs ONCE, then auto-skips)
This HF mirror is synced to the Zenodo **v7 (Nov 2025)** training archive
= 149 subjects, stored as 149 `.parquet` files (~27 GB total).

The cell checks Drive first: if the parquet files already exist, it SKIPS
the download, so `Runtime -> Run all` is safe every session.

To try it fast first, set `DEMO_ONLY = True` to grab just the first ~10
patients instead of all 149.

In [ ]:
import glob
from huggingface_hub import snapshot_download

DEMO_ONLY = False   # True = download only the first ~10 patients (quick test)

existing = glob.glob(os.path.join(PROJECT, 'data', '*.parquet'))
if existing:
    print(f'Data already present ({len(existing)} parquet files) - skipping download.')
else:
    print('No data found - downloading (one-time slow step)...')
    patterns = ['data/train-0000*.parquet'] if DEMO_ONLY else ['data/*.parquet']
    snapshot_download(
        repo_id='hugging-science/isles24-stroke',
        repo_type='dataset',
        local_dir=PROJECT,
        allow_patterns=patterns,
    )
    print('Download complete.')

In [ ]:
# --- See what downloaded ---
shards = sorted(glob.glob(os.path.join(PROJECT, 'data', '*.parquet')))
print(f'{len(shards)} patient files (shards) found.')
print('First few:', [os.path.basename(s) for s in shards[:3]])

## Load one patient
Each imaging column in the table is a little package: `bytes` (the scan
file, compressed) + `path` (its original name). The helper below writes
those bytes to a temporary file and loads it with nibabel.

We only decode the four blood-flow maps + the lesion mask. We SKIP the
big ones (ncct / cta / ctp / dwi / adc) — especially `ctp`, the giant raw
4D scan — to keep memory low.

In [ ]:
import pyarrow.parquet as pq
import nibabel as nib, numpy as np, tempfile

def load_nifti(struct):
    """struct = {'bytes':..., 'path':...} from the table -> 3D float array (or None)."""
    if not struct or struct.get('bytes') is None:
        return None
    raw, path = struct['bytes'], (struct.get('path') or 'img.nii.gz')
    suffix = '.nii.gz' if path.endswith('.gz') else '.nii'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tf:
        tf.write(raw); tmp = tf.name
    try:
        return np.asanyarray(nib.load(tmp).dataobj).astype(np.float32)
    finally:
        os.remove(tmp)

def load_patient(shard_path, map_keys=('cbf', 'cbv', 'mtt', 'tmax')):
    """Read one parquet shard -> (meta dict, {map: volume}, mask)."""
    row = pq.read_table(shard_path).slice(0, 1).to_pylist()[0]
    meta = {
        'subject_id': row['subject_id'], 'age': row.get('age'),
        'sex': row.get('sex'), 'nihss_admission': row.get('nihss_admission'),
    }
    volumes = {k: load_nifti(row[k]) for k in map_keys}
    mask = load_nifti(row['lesion_mask'])
    return meta, volumes, mask

meta, volumes, mask = load_patient(shards[0])
print('subject:', meta['subject_id'], '| age:', meta['age'], '| sex:', meta['sex'])
for k, v in volumes.items():
    print(f'{k}: {None if v is None else v.shape}')
print('lesion_mask:', None if mask is None else f"{mask.shape}, lesion voxels = {int((mask > 0).sum())}")

## Visualize the maps (sanity check)
A quick picture confirms the scans loaded correctly and the lesion
outline sits on brain tissue. It auto-picks the slice with the most
brain signal (not a blind middle slice) and ignores empty/NaN voxels.

In [ ]:
import matplotlib.pyplot as plt

def best_slice_idx(vol):
    v = np.nan_to_num(vol)
    counts = [np.count_nonzero(v[:, :, z]) for z in range(v.shape[2])]
    return int(np.argmax(counts))

if not volumes or all(v is None for v in volumes.values()):
    print("No volumes loaded - check the load cell above.")
else:
    items = [(k, v) for k, v in volumes.items() if v is not None]
    n = len(items)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (k, vol) in zip(axes, items):
        z = best_slice_idx(vol)
        ax.imshow(np.nan_to_num(vol[:, :, z]).T, cmap='jet', origin='lower')
        if mask is not None and mask.shape == vol.shape:
            ax.contour(np.nan_to_num(mask[:, :, z]).T, colors='white', linewidths=0.6)
        ax.set_title(f'{k.upper()} (z={z})'); ax.axis('off')
    plt.tight_layout(); plt.show()

## Extract left-vs-right asymmetry features
Split the volume at the midline (x axis) and compare blood-flow stats
between hemispheres. In a stroke, one side is starved of blood, so these
left-vs-right differences are your interpretable signal.

In [ ]:
import pandas as pd

def asymmetry_features(vol):
    x = vol.shape[0] // 2
    left, right = vol[:x], vol[x:]
    left = left[np.isfinite(left)]; right = right[np.isfinite(right)]
    lm, rm = float(np.mean(left)), float(np.mean(right))
    return {
        'left_mean': lm, 'right_mean': rm,
        'abs_diff': abs(lm - rm),
        'asym_ratio': abs(lm - rm) / (abs(lm) + abs(rm) + 1e-6),
        'left_p90': float(np.percentile(left, 90)),
        'right_p90': float(np.percentile(right, 90)),
    }

row = {'subject_id': meta['subject_id'], 'age': meta['age'], 'sex': meta['sex']}
for k, vol in volumes.items():
    if vol is None:
        continue
    for fname, val in asymmetry_features(vol).items():
        row[f'{k}_{fname}'] = val
if mask is not None:
    row['lesion_voxels'] = int(np.sum(mask > 0))
    row['has_lesion'] = int(np.sum(mask > 0) > 0)

df = pd.DataFrame([row])
df

In [ ]:
# --- Save the feature row to Drive (this is the small file you keep) ---
out = os.path.join(PROJECT, 'features_demo.csv')
df.to_csv(out, index=False)
print('Saved:', out)

## Next steps (Week 2)
- Loop over all 149 shards, ONE at a time, collecting one feature row per
  patient -> full `features.csv`. Keep only the small numbers; let each big
  scan be freed before the next, so RAM stays flat.
- Add more features: extra percentiles, % of tissue past a Tmax threshold.
- Week 3: train a baseline + models on the table, using a PATIENT-LEVEL split.

Loop skeleton for Week 2:

```python
rows = []
for sh in shards:
    meta, volumes, mask = load_patient(sh)
    r = {'subject_id': meta['subject_id'], 'age': meta['age'], 'sex': meta['sex']}
    for k, vol in volumes.items():
        if vol is None: continue
        for fname, val in asymmetry_features(vol).items():
            r[f'{k}_{fname}'] = val
    if mask is not None:
        r['lesion_voxels'] = int((mask > 0).sum())
        r['has_lesion'] = int((mask > 0).sum() > 0)
    rows.append(r)
    del volumes, mask        # free memory before next patient
pd.DataFrame(rows).to_csv(os.path.join(PROJECT, 'features.csv'), index=False)
```